# 📖 Notebook 4: TLS and mTLS

📖 **Source**: [Hello Interview – Networking Essentials](https://www.hellointerview.com/learn/system-design/01-foundations/networking-essentials)

When you see the 🔒 in your browser, that's **TLS** (Transport Layer Security) at work. It encrypts everything between your browser and the server so nobody in the middle can read or tamper with your data. **mTLS** goes one step further: both sides prove their identity.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why encryption matters (man-in-the-middle attacks)
- How TLS works (the handshake, certificates, encryption)
- How to generate and use TLS certificates
- What mTLS is and when to use it (microservices zero-trust)
- Certificate Authorities and trust chains

## 🛠️ Setup

First generate TLS certificates, then start Docker:

```bash
cd 01-foundations/networking-essentials

# Generate certificates (CA + server + client)
bash nginx/generate_certs.sh

# Start nginx + backends
docker compose up -d --build
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import ssl
import socket
import os
import json
import requests
import time
from datetime import datetime, timezone
from cryptography import x509
from cryptography.x509.oid import NameOID
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import rsa

# Path to our generated certificates
CERTS_DIR = os.path.join(os.path.dirname(os.getcwd()), "nginx", "certs")
if not os.path.isdir(CERTS_DIR):
    # Try relative to notebook location
    CERTS_DIR = os.path.join(os.getcwd(), "..", "nginx", "certs")
    CERTS_DIR = os.path.abspath(CERTS_DIR)

if os.path.isdir(CERTS_DIR):
    print(f"✅ Certificates found at {CERTS_DIR}")
    print(f"   Files: {os.listdir(CERTS_DIR)}")
else:
    print("❌ Certificates not found. Run: bash nginx/generate_certs.sh")

---
## Part 1: Why Encryption Matters

Without TLS, anyone between you and the server can:

1. **Read your data** (passwords, credit cards, private messages)
2. **Modify your data** (inject malware, change bank transfers)
3. **Impersonate the server** (trick you into sending data to them)

This is called a **man-in-the-middle (MITM) attack**:

```
Without TLS:
You ──plain text──→ [☠️ Attacker reads everything] ──→ Server

With TLS:
You ──🔒 encrypted──→ [☠️ Attacker sees gibberish] ──→ Server
```

Let's see the difference!

In [ ]:
# === HTTP vs HTTPS: What an attacker sees ===

# Simulated sensitive data
login_data = {
    "username": "alice",
    "password": "super_secret_123",
    "credit_card": "4111-1111-1111-1111"
}

# What goes over the wire with HTTP (plain text)
http_wire = f"""POST /login HTTP/1.1\r\nHost: bank.com\r\nContent-Type: application/json\r\n\r\n{json.dumps(login_data)}"""

print("=== What an attacker sees WITHOUT TLS (HTTP) ===")
print()
print(http_wire)
print()
print("☠️  Everything is readable — passwords, credit cards, everything!")
print()

# What goes over the wire with HTTPS (encrypted)
import hashlib
encrypted_simulation = hashlib.sha256(http_wire.encode()).hexdigest()

print("=== What an attacker sees WITH TLS (HTTPS) ===")
print()
print(f"17 03 03 01 2c {encrypted_simulation[:40]}...")
print("(binary gibberish — the actual encrypted TLS record)")
print()
print("🔒 The attacker can see that traffic is flowing, but not what it contains.")
print("   They can't even tell it's a login request!")

---
## Part 2: How TLS Works

### The TLS Handshake

Before encrypted communication begins, client and server perform a **TLS handshake**:

```
Client                              Server
  │                                    │
  │── ClientHello ────────────────────→│  "I support TLS 1.3, these ciphers..."
  │                                    │
  │←── ServerHello + Certificate ─────│  "Let's use TLS 1.3 with AES-256.
  │                                    │   Here's my certificate to prove I'm real."
  │                                    │
  │── (verify cert, key exchange) ────→│  "Your cert checks out. Here's our
  │                                    │   shared secret for encryption."
  │                                    │
  │←═══════ Encrypted data ══════════→│  🔒 All traffic is now encrypted
```

The certificate is crucial — it's how you know you're **really talking to the right server** and not an impersonator.

In [ ]:
# === Examining a Real TLS Certificate ===
# Let's look at what's inside a TLS certificate for a real website.

import ssl
import socket

def inspect_tls_certificate(hostname, port=443):
    """Connect to a server and inspect its TLS certificate."""
    context = ssl.create_default_context()
    with socket.create_connection((hostname, port), timeout=5) as sock:
        with context.wrap_socket(sock, server_hostname=hostname) as tls_sock:
            cert = tls_sock.getpeercert()
            cipher = tls_sock.cipher()
            version = tls_sock.version()
            return cert, cipher, version

hostname = "github.com"
cert, cipher, tls_version = inspect_tls_certificate(hostname)

print(f"🔒 TLS Certificate for {hostname}:\n")
print(f"  TLS Version:  {tls_version}")
print(f"  Cipher Suite: {cipher[0]}")
print(f"  Key Size:     {cipher[2]} bits\n")

# Parse the certificate fields
subject = dict(x[0] for x in cert['subject'])
issuer = dict(x[0] for x in cert['issuer'])

print(f"  Subject (who the cert belongs to):")
for key, value in subject.items():
    print(f"    {key}: {value}")

print(f"\n  Issuer (who signed the cert):")
for key, value in issuer.items():
    print(f"    {key}: {value}")

print(f"\n  Valid from: {cert['notBefore']}")
print(f"  Valid to:   {cert['notAfter']}")

print("\n💡 The 'issuer' is a Certificate Authority (CA) that vouches for this server.")
print("   Your browser trusts a list of CAs, and if the chain checks out, you see 🔒")

### Certificate Trust Chain

How does your browser know to trust a certificate? Through a **chain of trust**:

```
Root CA (pre-installed in your OS/browser)
  └── signs → Intermediate CA
                └── signs → Server Certificate (e.g., github.com)
```

Your browser has ~150 pre-installed **Root CAs** it trusts. When a server presents a certificate, the browser walks up the chain to verify it reaches a trusted root.

For our lab, we created our **own CA** — it's not trusted by browsers, but we can tell our code to trust it explicitly.

In [ ]:
# === Examining Our Lab's Self-Signed Certificates ===

def load_and_inspect_cert(cert_path):
    """Load a PEM certificate file and display its contents."""
    with open(cert_path, "rb") as f:
        cert = x509.load_pem_x509_certificate(f.read())

    print(f"  Subject:    {cert.subject.rfc4514_string()}")
    print(f"  Issuer:     {cert.issuer.rfc4514_string()}")
    print(f"  Valid from: {cert.not_valid_before_utc}")
    print(f"  Valid to:   {cert.not_valid_after_utc}")
    print(f"  Serial:     {cert.serial_number}")

    # Check if it's self-signed (subject == issuer)
    is_self_signed = cert.subject == cert.issuer
    print(f"  Self-signed: {is_self_signed}")
    return cert

try:
    print("=== Our Certificate Authority (CA) ===")
    ca_cert = load_and_inspect_cert(os.path.join(CERTS_DIR, "ca.crt"))

    print("\n=== Our Server Certificate ===")
    server_cert = load_and_inspect_cert(os.path.join(CERTS_DIR, "server.crt"))

    print("\n=== Our Client Certificate (for mTLS) ===")
    client_cert = load_and_inspect_cert(os.path.join(CERTS_DIR, "client.crt"))

    print("\n💡 The CA is self-signed (root of trust).")
    print("   Server and client certs are signed BY the CA.")
    print("   Anyone who trusts the CA will trust these certs.")
except FileNotFoundError:
    print("❌ Certificates not found. Run: bash nginx/generate_certs.sh")

---
## Part 3: TLS in Action

Our nginx is configured with TLS on port 8443. Let's connect to it!

In [ ]:
# === Connecting to Our HTTPS Server ===

# First, try WITHOUT trusting our CA (this should fail or warn)
print("=== Attempt 1: Without trusting our CA ===")
try:
    r = requests.get("https://localhost:8443/", timeout=3)
    print(f"  Got response: {r.status_code}")
except requests.exceptions.SSLError as e:
    print(f"  ❌ SSL Error: Certificate not trusted!")
    print(f"     This is CORRECT — our CA is not in the system trust store.")
except Exception as e:
    print(f"  ⚠️ Error: {type(e).__name__}: {e}")

print()

# Now, trust our CA explicitly
print("=== Attempt 2: Trusting our CA certificate ===")
try:
    ca_cert_path = os.path.join(CERTS_DIR, "ca.crt")
    r = requests.get("https://localhost:8443/", verify=ca_cert_path, timeout=3)
    print(f"  ✅ Got response: {r.status_code}")
    data = r.json()
    print(f"  Server: {data['server']}")
    print(f"  Message: {data['message']}")
except Exception as e:
    print(f"  ⚠️ Error: {type(e).__name__}: {e}")

print()

# Skip verification entirely (NEVER do this in production!)
print("=== Attempt 3: Skipping verification (INSECURE — never in production!) ===")
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
r = requests.get("https://localhost:8443/", verify=False, timeout=3)
print(f"  ⚠️ Got response: {r.status_code} (but we didn't verify the server!)")
print(f"  This is dangerous — we could be talking to an impersonator!")

In [ ]:
# === TLS Handshake Details ===
# Let's see exactly what happens during the TLS handshake with our server.

import ssl

def inspect_tls_connection(host, port, ca_cert_path):
    """Establish a TLS connection and inspect the negotiated parameters."""
    context = ssl.SSLContext(ssl.PROTOCOL_TLS_CLIENT)
    context.load_verify_locations(ca_cert_path)

    with socket.create_connection((host, port), timeout=5) as sock:
        with context.wrap_socket(sock, server_hostname=host) as tls_sock:
            return {
                "tls_version": tls_sock.version(),
                "cipher": tls_sock.cipher(),
                "compression": tls_sock.compression(),
                "peer_cert": tls_sock.getpeercert(),
            }

try:
    ca_path = os.path.join(CERTS_DIR, "ca.crt")
    info = inspect_tls_connection("localhost", 8443, ca_path)

    print("🔒 TLS Connection Details:\n")
    print(f"  TLS Version:   {info['tls_version']}")
    print(f"  Cipher Suite:  {info['cipher'][0]}")
    print(f"  Key Size:      {info['cipher'][2]} bits")
    print(f"  Compression:   {info['compression'] or 'None (good — avoids CRIME attack)'}")

    cert = info['peer_cert']
    subject = dict(x[0] for x in cert['subject'])
    print(f"\n  Server Certificate:")
    print(f"    CN (Common Name): {subject.get('commonName', 'N/A')}")
    print(f"    Valid until: {cert['notAfter']}")

    print("\n💡 The cipher suite determines how your data is encrypted.")
    print("   TLS 1.3 is the latest and most secure version.")
except Exception as e:
    print(f"⚠️  Could not connect: {e}")
    print("   Make sure certs are generated and docker-compose is running.")

---
## Part 4: mTLS — Mutual TLS

Regular TLS is **one-way**: the client verifies the server. But the server doesn't verify the client — anyone can connect.

**mTLS (Mutual TLS)** adds the other direction: the server also requires the client to present a certificate.

```
Regular TLS:                          mTLS:
Client ──→ "Show me your cert"        Client ──→ "Show me your cert"
Server ──→ 📜 (server cert)           Server ──→ 📜 (server cert)
Client ──→ ✅ "I trust you"           Client ──→ ✅ "I trust you"
                                      Server ──→ "Now show me YOUR cert"
                                      Client ──→ 📜 (client cert)
                                      Server ──→ ✅ "I trust you too"
```

### Why Use mTLS?

In a **zero-trust architecture** (common in microservices), every service must prove its identity:

```
User Service ──mTLS──→ Auth Service     (both verify each other)
Order Service ──mTLS──→ Payment Service  (both verify each other)
Random Attacker ──→ ❌ Rejected!         (no valid client cert)
```

This prevents unauthorized services from talking to your internal APIs, even if they're on the same network.

In [ ]:
# === mTLS: Server Requires Client Certificate ===
# Our nginx listens on port 9443 with mTLS enabled.

ca_path = os.path.join(CERTS_DIR, "ca.crt")
client_cert_path = os.path.join(CERTS_DIR, "client.crt")
client_key_path = os.path.join(CERTS_DIR, "client.key")

# Attempt 1: Connect WITHOUT a client certificate
print("=== Attempt 1: No client certificate ===")
try:
    r = requests.get("https://localhost:9443/", verify=ca_path, timeout=3)
    print(f"  Response: {r.status_code} — {r.text[:100]}")
except requests.exceptions.SSLError as e:
    print(f"  ❌ Rejected! Server requires a client certificate.")
    print(f"     This is mTLS working as intended.")
except Exception as e:
    print(f"  ❌ Error: {type(e).__name__}: {e}")

print()

# Attempt 2: Connect WITH our client certificate
print("=== Attempt 2: With valid client certificate ===")
try:
    r = requests.get(
        "https://localhost:9443/",
        verify=ca_path,
        cert=(client_cert_path, client_key_path),  # client cert + key
        timeout=3
    )
    print(f"  ✅ Response: {r.status_code}")
    data = r.json()
    print(f"  Server: {data['server']}")
    print(f"  Message: {data['message']}")
except Exception as e:
    print(f"  ⚠️ Error: {type(e).__name__}: {e}")

print()
print("💡 mTLS ensures BOTH sides prove their identity.")
print("   Without a valid client cert, the server refuses the connection.")
print("   This is how microservices authenticate each other in zero-trust networks.")

---
## Part 5: Creating Certificates with Python

Let's understand certificates by creating them from scratch with the `cryptography` library.

In [ ]:
# === Creating a Certificate Authority from Scratch ===
import ipaddress
from datetime import timedelta

# Step 1: Generate a private key for our CA
ca_private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048,  # 2048 bits is the practical minimum today
)
print("Step 1: Generated CA private key (2048-bit RSA)")

# Step 2: Create a self-signed CA certificate
ca_name = x509.Name([
    x509.NameAttribute(NameOID.COMMON_NAME, "My Lab CA"),
    x509.NameAttribute(NameOID.ORGANIZATION_NAME, "Learning Labs"),
])

now = datetime.now(timezone.utc)
ca_cert = (
    x509.CertificateBuilder()
    .subject_name(ca_name)
    .issuer_name(ca_name)               # self-signed: issuer == subject
    .public_key(ca_private_key.public_key())
    .serial_number(x509.random_serial_number())
    .not_valid_before(now)
    .not_valid_after(now + timedelta(days=365))
    .add_extension(
        # BasicConstraints with ca=True is what makes this cert allowed to sign others
        x509.BasicConstraints(ca=True, path_length=None),
        critical=True,
    )
    .sign(ca_private_key, hashes.SHA256())
)
print("Step 2: Created self-signed CA certificate")
print(f"         Subject: {ca_cert.subject.rfc4514_string()}")
print(f"         Issuer:  {ca_cert.issuer.rfc4514_string()}")

# Step 3: Generate a server key + certificate signed by our CA
server_private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
print("\nStep 3: Generated server private key")

server_name = x509.Name([
    x509.NameAttribute(NameOID.COMMON_NAME, "my-api.example.com"),
    x509.NameAttribute(NameOID.ORGANIZATION_NAME, "My API Service"),
])

server_cert = (
    x509.CertificateBuilder()
    .subject_name(server_name)
    .issuer_name(ca_name)               # signed BY our CA, not self-signed
    .public_key(server_private_key.public_key())
    .serial_number(x509.random_serial_number())
    .not_valid_before(now)
    .not_valid_after(now + timedelta(days=90))
    .add_extension(
        # SubjectAlternativeName lists every name/IP the cert is valid for.
        # Modern TLS clients ignore the Common Name and only look at the SAN.
        x509.SubjectAlternativeName([
            x509.DNSName("my-api.example.com"),
            x509.DNSName("localhost"),
            x509.IPAddress(ipaddress.IPv4Address("127.0.0.1")),
        ]),
        critical=False,
    )
    .sign(ca_private_key, hashes.SHA256())   # signed with the CA's private key
)
print("Step 4: Created server certificate (signed by our CA)")
print(f"         Subject: {server_cert.subject.rfc4514_string()}")
print(f"         Issuer:  {server_cert.issuer.rfc4514_string()}")

print("\n=== Certificate Chain ===")
print(f"  Root CA: {ca_cert.subject.rfc4514_string()} (self-signed, trusted)")
print(f"    +-- Server: {server_cert.subject.rfc4514_string()} (signed by CA)")
print("\nIn production you would use a real public CA like Let's Encrypt for")
print("public sites, or run your own internal CA for service-to-service certs.")


---
## Part 6: TLS Performance Impact

TLS adds security but also adds **latency** (the handshake) and **CPU cost** (encryption/decryption). Let's measure it.

In [ ]:
# === HTTP vs HTTPS Performance ===
# We measure two scenarios so you can see *where* TLS adds cost:
#   1. Reused connection (one Session) -- the handshake happens ONCE.
#   2. Fresh connection per request    -- the handshake happens EVERY time.
# Real apps almost always reuse connections, but seeing the difference makes it
# obvious why connection pooling matters so much when TLS is enabled.

NUM_REQUESTS = 50
ca_path = os.path.join(CERTS_DIR, "ca.crt")

def time_session_requests(url, **kw):
    """One Session => one TCP+TLS connection reused for all requests."""
    s = requests.Session()
    start = time.perf_counter()
    for _ in range(NUM_REQUESTS):
        s.get(url, **kw)
    elapsed = time.perf_counter() - start
    s.close()
    return elapsed

def time_fresh_connection(url, **kw):
    """New Session each call => fresh TCP handshake (and TLS handshake if HTTPS) every time."""
    start = time.perf_counter()
    for _ in range(NUM_REQUESTS):
        with requests.Session() as s:
            s.get(url, **kw)
    return time.perf_counter() - start

http_url, https_url = "http://localhost:8080/", "https://localhost:8443/"

http_reused  = time_session_requests(http_url)
https_reused = time_session_requests(https_url, verify=ca_path)
http_fresh   = time_fresh_connection(http_url)
https_fresh  = time_fresh_connection(https_url, verify=ca_path)

print(f"Performance comparison ({NUM_REQUESTS} requests each):\n")
print(f"  Connection reused (one TLS handshake total):")
print(f"    HTTP : {http_reused*1000:6.0f} ms  ({http_reused/NUM_REQUESTS*1000:.2f} ms/req)")
print(f"    HTTPS: {https_reused*1000:6.0f} ms  ({https_reused/NUM_REQUESTS*1000:.2f} ms/req)")
print(f"    Encryption-only overhead: {(https_reused-http_reused)/http_reused*100:+.0f}%")

print(f"\n  Fresh connection per request (handshake EVERY time):")
print(f"    HTTP : {http_fresh*1000:6.0f} ms  ({http_fresh/NUM_REQUESTS*1000:.2f} ms/req)")
print(f"    HTTPS: {https_fresh*1000:6.0f} ms  ({https_fresh/NUM_REQUESTS*1000:.2f} ms/req)")
print(f"    Handshake-included overhead: {(https_fresh-http_fresh)/http_fresh*100:+.0f}%")

print("\nThe cost of TLS is mostly in the *handshake*, not in encrypting bytes.")
print("Modern CPUs have AES acceleration, so per-byte encryption is nearly free.")
print("That is why HTTP keep-alive, HTTP/2 multiplexing, and TLS session resumption")
print("matter so much -- they avoid repeating the handshake.")
print("\nNote: on localhost the absolute differences are tiny and noise dominates")
print("(you may even see a 'negative' overhead). On a real network with real")
print("round-trip latency, the handshake cost is much more visible.")


---
## Part 7: TLS Termination at the Load Balancer

In real deployments you almost never put TLS certificates on every backend. Instead, the **load balancer / reverse proxy terminates TLS** and talks plain HTTP to the backends:

```
Internet --HTTPS (TLS)--> Load Balancer --HTTP (plaintext)--> backend1
                              |                           +-> backend2
                              |                           +-> backend3
                              +- holds the cert + private key
```

### Why this is the common pattern

- **One certificate to manage** instead of one per backend.
- **Cheap rotation**: Reload the LB, no backend restarts.
- **CPU offload**: TLS work happens on a few LBs (often with hardware acceleration), not on every app server.
- **Easier inspection**: WAFs, rate limiters, and access logs see plaintext HTTP.

### When you should NOT terminate at the edge

- **Zero-trust internal networks**: re-encrypt with **mTLS** between LB and backends so a compromised internal host cannot sniff traffic. Service meshes like **Istio** and **Linkerd** automate exactly this.
- **Compliance** (HIPAA, PCI): sometimes requires end-to-end encryption all the way to the application.
- **Direct client-to-backend protocols** (e.g., gRPC with client certs) where the LB only forwards TCP (Layer 4 passthrough).

### What our lab is doing

Our nginx is the TLS terminator. It listens on `:8443` (TLS) and `:9443` (mTLS), then proxies to the Flask backends over plain HTTP on the Docker network -- exactly the pattern above.


---
## 🎓 Key Takeaways

1. **TLS encrypts** all traffic between client and server — always use HTTPS for public traffic
2. The **TLS handshake** adds ~1-2 round trips of latency (TLS 1.3 reduces this)
3. **Certificates** prove identity — signed by a trusted Certificate Authority
4. **mTLS** adds client certificates so both sides prove their identity
5. Use **mTLS** for service-to-service communication in zero-trust architectures
6. TLS performance overhead is small (~5-15%) and worth the security

### Interview Tips
- Always mention **HTTPS** when discussing public-facing APIs
- Bring up **mTLS** when discussing microservices security or zero-trust
- Remember: TLS encrypts the *content* but not the fact that a connection exists
- **Certificate rotation** is an operational concern — mention it for reliability discussions
- TLS termination at the load balancer is common (the LB decrypts, forwards plain HTTP internally)